In [1]:
from google.colab import files

uploaded = files.upload()

Saving placement_predict_50k_adjusted.csv to placement_predict_50k_adjusted.csv


In [2]:
import os

print(os.listdir())

['.config', 'placement_predict_50k_adjusted.csv', 'sample_data']


In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("placement_predict_50k_adjusted.csv")

print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. TARGET VARIABLE
# ============================================================

# 0 = Not Placed
# 1 = Placed

y = df["PlacementStatus"]


# ============================================================
# 3. SELECT PREDICTOR VARIABLES
# ============================================================

features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

X = df[features].copy()


# ============================================================
# 4. CATEGORICAL VARIABLES
# ============================================================

categorical_features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",
    "ExtraCurricular"
]


# ============================================================
# 5. NUMERICAL VARIABLES
# ============================================================

numerical_features = [
    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore"
]


# ============================================================
# 6. HANDLE MISSING VALUES
# ============================================================

for col in numerical_features:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_features:
    X[col] = X[col].fillna(X[col].mode()[0])


# ============================================================
# 7. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 8. PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),

        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)


# ============================================================
# 9. BINOMIAL LOGISTIC REGRESSION
# ============================================================

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)


# ============================================================
# 10. TRAIN MODEL
# ============================================================

logistic_model.fit(X_train, y_train)


# ============================================================
# 11. PREDICTION
# ============================================================

y_pred = logistic_model.predict(X_test)

y_probability = logistic_model.predict_proba(X_test)[:, 1]


# ============================================================
# 12. MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\n============================================")
print("BINOMIAL LOGISTIC REGRESSION RESULTS")
print("============================================")

print("\nAccuracy:")
print(round(accuracy, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Placed", "Placed"]
    )
)

print("\nROC-AUC:")
print(round(roc_auc_score(y_test, y_probability), 4))

Dataset Shape: (50000, 21)

Columns:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'PlacementStatus', 'IsAnomaly']

BINOMIAL LOGISTIC REGRESSION RESULTS

Accuracy:
0.7899

Confusion Matrix:
[[4228 1022]
 [1079 3671]]

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.80      0.81      0.80      5250
      Placed       0.78      0.77      0.78      4750

    accuracy                           0.79     10000
   macro avg       0.79      0.79      0.79     10000
weighted avg       0.79      0.79      0.79     10000


ROC-AUC:
0.8769


In [8]:
l1_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=1.0,
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

l1_model.fit(X_train, y_train)

y_pred_l1 = l1_model.predict(X_test)

y_prob_l1 = l1_model.predict_proba(X_test)[:, 1]

accuracy_l1 = accuracy_score(y_test, y_pred_l1)

print("\n============================================")
print("L1 LOGISTIC REGRESSION RESULTS")
print("============================================")

print("\nAccuracy:")
print(round(accuracy_l1, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_l1))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_l1,
        target_names=["Not Placed", "Placed"]
    )
)

print("\nROC-AUC:")
print(round(roc_auc_score(y_test, y_prob_l1), 4))


L1 LOGISTIC REGRESSION RESULTS

Accuracy:
0.7903

Confusion Matrix:
[[4229 1021]
 [1076 3674]]

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.80      0.81      0.80      5250
      Placed       0.78      0.77      0.78      4750

    accuracy                           0.79     10000
   macro avg       0.79      0.79      0.79     10000
weighted avg       0.79      0.79      0.79     10000


ROC-AUC:
0.8769


In [9]:
l2_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                C=1.0,
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

l2_model.fit(X_train, y_train)

y_pred_l2 = l2_model.predict(X_test)

y_prob_l2 = l2_model.predict_proba(X_test)[:, 1]

accuracy_l2 = accuracy_score(y_test, y_pred_l2)

print("\n============================================")
print("L2 LOGISTIC REGRESSION RESULTS")
print("============================================")

print("\nAccuracy:")
print(round(accuracy_l2, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_l2))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_l2,
        target_names=["Not Placed", "Placed"]
    )
)

print("\nROC-AUC:")
print(round(roc_auc_score(y_test, y_prob_l2), 4))


L2 LOGISTIC REGRESSION RESULTS

Accuracy:
0.7899

Confusion Matrix:
[[4228 1022]
 [1079 3671]]

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.80      0.81      0.80      5250
      Placed       0.78      0.77      0.78      4750

    accuracy                           0.79     10000
   macro avg       0.79      0.79      0.79     10000
weighted avg       0.79      0.79      0.79     10000


ROC-AUC:
0.8769


In [10]:
elastic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                l1_ratio=0.5,
                C=1.0,
                max_iter=5000,
                random_state=42
            )
        )
    ]
)

elastic_model.fit(X_train, y_train)

y_pred_elastic = elastic_model.predict(X_test)

y_prob_elastic = elastic_model.predict_proba(X_test)[:, 1]

accuracy_elastic = accuracy_score(
    y_test,
    y_pred_elastic
)

print("\n============================================")
print("ELASTIC NET LOGISTIC REGRESSION RESULTS")
print("============================================")

print("\nAccuracy:")
print(round(accuracy_elastic, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_elastic))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_elastic,
        target_names=["Not Placed", "Placed"]
    )
)

print("\nROC-AUC:")
print(
    round(
        roc_auc_score(
            y_test,
            y_prob_elastic
        ),
        4
    )
)


ELASTIC NET LOGISTIC REGRESSION RESULTS

Accuracy:
0.7899

Confusion Matrix:
[[4228 1022]
 [1079 3671]]

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.80      0.81      0.80      5250
      Placed       0.78      0.77      0.78      4750

    accuracy                           0.79     10000
   macro avg       0.79      0.79      0.79     10000
weighted avg       0.79      0.79      0.79     10000


ROC-AUC:
0.8769


In [11]:
new_student = pd.DataFrame([
    {
        "Gender": "Male",
        "City": "Bangalore",
        "CollegeTier": "Tier1",
        "Stream": "Computer Science",
        "Specialisation": "Computer Science",
        "Hostel": "Yes",
        "HistoryOfBacklogs": "No",
        "CGPA": 8.65,
        "AttendancePercent": 92,
        "Internships": 2,
        "Projects": 4,
        "Workshops": 5,
        "Certifications": 4,
        "Publications": 1,
        "AptitudeTestScore": 85,
        "SoftSkillsRating": 8.5,
        "CodingTestScore": 88,
        "MockInterviewScore": 8.2,
        "ExtraCurricular": "Yes",
    }
])

prediction = elastic_model.predict(new_student)[0]

probability = elastic_model.predict_proba(new_student)[0, 1]

print("\n============================================")
print("NEW STUDENT PLACEMENT PREDICTION")
print("============================================")

if prediction == 1:
    print("Predicted Placement Status : PLACED")
elif prediction == 0:
    print("Predicted Placement Status : NOT PLACED")
else:
    print("Invalid prediction received.")

print(f"Probability of Placement    : {probability * 100:.2f}%")

print(f"Probability of Not Placed   : {(1 - probability) * 100:.2f}%")


NEW STUDENT PLACEMENT PREDICTION
Predicted Placement Status : PLACED
Probability of Placement    : 91.05%
Probability of Not Placed   : 8.95%


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [3, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [3, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
